In [ ]:
# ============================================================
# GIÁO TRÌNH: THỊ GIÁC MÁY: TỪ XỬ LÝ ẢNH ĐẾN HỌC SÂU
# BÀI CODE MINH HỌA: TÍNH ĐẶC TRƯNG HARALICK TỪ MA TRẬN GLCM
# Chương/Mục liên quan: Chương 5 - Đặc trưng kết cấu ảnh
# ============================================================

# ============================================================
# MÔ TẢ
# ============================================================

# Mục đích:
# - Minh họa cách xây dựng ma trận đồng xuất hiện mức xám GLCM.
# - Tính 14 đặc trưng Haralick từ ma trận xác suất p(i,j).
# - Tổng hợp đặc trưng theo 4 hướng để tạo vector Haralick 28 chiều.

# Sau khi chạy code, người học cần:
# 1. Hiểu được cách lượng tử hóa ảnh mức xám trước khi tính GLCM.
# 2. Quan sát được ma trận GLCM P(i,j) và ma trận xác suất p(i,j).
# 3. Hiểu cách 14 đặc trưng Haralick được tính từ p(i,j).
# 4. Hiểu cách lấy trung bình và độ lệch chuẩn theo nhiều hướng.

# Input:
# - Dữ liệu đầu vào: ảnh xám từ GitHub.
# - URL: https://github.com/lthavnu/cv-book/raw/main/images/wooden.png
# - Kiểu dữ liệu: ảnh xám.

# Output:
# - Kích thước ảnh đầu vào.
# - Các mức xám sau lượng tử hóa.
# - Ma trận GLCM đếm P(i,j) cho từng hướng.
# - Ma trận xác suất GLCM p(i,j) cho từng hướng.
# - 14 đặc trưng Haralick cho từng hướng.
# - Vector Haralick 28 chiều.

# Lưu ý:
# Đoạn code này được xây dựng với sự hỗ trợ của công cụ AI.
# Giảng viên đã đọc, kiểm tra và hiệu chỉnh nhằm bảo đảm tính chính xác,
# tính sư phạm và sự phù hợp với nội dung lý thuyết trong giáo trình.

# Người học không chỉ chạy code để xem kết quả, mà cần hiểu:
# - Code đang minh họa nội dung lý thuyết nào;
# - Các bước xử lý tương ứng với công thức hoặc thuật toán nào;
# - Kết quả đầu ra có hợp lý hay không;
# - Khi thay đổi tham số, kết quả thay đổi như thế nào.

# ============================================================
# 1. CÀI ĐẶT VÀ IMPORT THƯ VIỆN
# ============================================================

import cv2
import numpy as np
from urllib.request import urlopen


# ============================================================
# 2. TẢI DỮ LIỆU ĐẦU VÀO
# ============================================================

def read_gitlab_image_gray(url):
    resp = urlopen(url)
    img = np.asarray(bytearray(resp.read()), dtype=np.uint8)
    return cv2.imdecode(img, cv2.IMREAD_GRAYSCALE)


image_uri = "https://github.com/lthavnu/cv-book/raw/main/images/wooden.png"

I_gray = read_gitlab_image_gray(image_uri)

if I_gray is None:
    raise ValueError("Không đọc được ảnh từ URL.")


# ============================================================
# 3. HIỂN THỊ THÔNG TIN ẢNH ĐẦU VÀO
# ============================================================

print("Kích thước ảnh:", I_gray.shape)


# ============================================================
# 4. XỬ LÝ CHÍNH
# ============================================================

# ------------------------------------------------------------
# 4.1. Lượng tử hóa ảnh về 5 mức xám
# ------------------------------------------------------------

gray_levels = np.array([0, 64, 128, 192, 255], dtype=np.uint8)

# Công thức/thuật toán minh họa:
# - Chuyển giá trị mức xám ban đầu về chỉ số mức xám rời rạc {0, 1, 2, 3, 4}.
I_q_idx = np.rint(I_gray / 64.0).astype(np.int32)
I_q_idx = np.clip(I_q_idx, 0, 4)

# Ảnh lượng tử hóa theo đúng 5 mức xám.
I_q = gray_levels[I_q_idx]

print("Các mức xám sau lượng tử hóa:", np.unique(I_q))


# ------------------------------------------------------------
# 4.2. Hàm tính ma trận GLCM đếm P(i,j)
# ------------------------------------------------------------

def compute_glcm_count_matrix(I_level_idx, d=1, theta_deg=45, num_levels=5, symmetric=False):
    # I_level_idx : ảnh đã lượng tử hóa, giá trị thuộc {0,1,2,3,4}
    # d           : khoảng cách giữa hai điểm ảnh
    # theta_deg   : hướng xét lân cận, gồm 0, 45, 90, 135 độ
    # symmetric   : nếu True thì cộng thêm P(j,i)

    if theta_deg == 0:
        dr, dc = 0, d
    elif theta_deg == 45:
        dr, dc = -d, d
    elif theta_deg == 90:
        dr, dc = -d, 0
    elif theta_deg == 135:
        dr, dc = -d, -d
    else:
        raise ValueError("theta_deg chỉ hỗ trợ: 0, 45, 90, 135")

    H, W = I_level_idx.shape
    P = np.zeros((num_levels, num_levels), dtype=np.float64)

    # Công thức/thuật toán minh họa:
    # - Với mỗi cặp điểm ảnh cách nhau d theo hướng theta,
    #   tăng P(i,j) lên 1 nếu điểm thứ nhất có mức i và điểm thứ hai có mức j.
    for r in range(H):
        for c in range(W):
            rr = r + dr
            cc = c + dc

            if 0 <= rr < H and 0 <= cc < W:
                i = I_level_idx[r, c]
                j = I_level_idx[rr, cc]
                P[i, j] += 1

                if symmetric:
                    P[j, i] += 1

    return P


# ------------------------------------------------------------
# 4.3. Hàm tính 14 đặc trưng Haralick từ p(i,j)
# ------------------------------------------------------------

def safe_log(x, eps=1e-12):
    return np.log(x + eps)


def haralick_14_features_from_p(p):
    Ng = p.shape[0]

    i_idx = np.arange(Ng)
    j_idx = np.arange(Ng)
    I, J = np.meshgrid(i_idx, j_idx, indexing='ij')

    # p_x(i), p_y(j)
    p_x = np.sum(p, axis=1)
    p_y = np.sum(p, axis=0)

    # mu_x, mu_y
    mu_x = np.sum(i_idx * p_x)
    mu_y = np.sum(j_idx * p_y)

    # sigma_x, sigma_y
    sigma_x = np.sqrt(np.sum(((i_idx - mu_x) ** 2) * p_x))
    sigma_y = np.sqrt(np.sum(((j_idx - mu_y) ** 2) * p_y))

    # p_{x+y}(k), k = 0..2(Ng-1)
    p_x_plus_y = np.zeros(2 * Ng - 1, dtype=np.float64)
    for k in range(2 * Ng - 1):
        p_x_plus_y[k] = np.sum(p[(I + J) == k])

    # p_{x-y}(k), k = 0..Ng-1, với k = |i-j|
    p_x_minus_y = np.zeros(Ng, dtype=np.float64)
    for k in range(Ng):
        p_x_minus_y[k] = np.sum(p[np.abs(I - J) == k])

    # HXY, HX, HY
    HXY = -np.sum(p * safe_log(p))
    HX  = -np.sum(p_x * safe_log(p_x))
    HY  = -np.sum(p_y * safe_log(p_y))

    # HXY1, HXY2
    px_py = np.outer(p_x, p_y)
    HXY1 = -np.sum(p * safe_log(px_py))
    HXY2 = -np.sum(px_py * safe_log(px_py))

    # h1: Angular Second Moment
    h1 = np.sum(p ** 2)

    # h2: Contrast
    h2 = np.sum(((I - J) ** 2) * p)

    # h3: Correlation
    if sigma_x > 1e-12 and sigma_y > 1e-12:
        h3 = np.sum(((I - mu_x) * (J - mu_y) * p) / (sigma_x * sigma_y))
    else:
        h3 = 0.0

    # h4: Variance
    mu = np.sum(I * p)
    h4 = np.sum(((I - mu) ** 2) * p)

    # h5: Inverse Difference Moment
    h5 = np.sum(p / (1.0 + (I - J) ** 2))

    # h6: Sum Average
    k_sum = np.arange(2 * Ng - 1)
    h6 = np.sum(k_sum * p_x_plus_y)

    # h7: Sum Variance
    h7 = np.sum(((k_sum - h6) ** 2) * p_x_plus_y)

    # h8: Sum Entropy
    h8 = -np.sum(p_x_plus_y * safe_log(p_x_plus_y))

    # h9: Entropy
    h9 = HXY

    # h10: Difference Variance
    k_diff = np.arange(Ng)
    mean_diff = np.sum(k_diff * p_x_minus_y)
    h10 = np.sum(((k_diff - mean_diff) ** 2) * p_x_minus_y)

    # h11: Difference Entropy
    h11 = -np.sum(p_x_minus_y * safe_log(p_x_minus_y))

    # h12: Information Measure of Correlation 1
    denom = max(HX, HY)
    h12 = (HXY - HXY1) / denom if denom > 1e-12 else 0.0

    # h13: Information Measure of Correlation 2
    h13 = np.sqrt(max(0.0, 1.0 - np.exp(-2.0 * (HXY2 - HXY))))

    # h14: Maximal Correlation Coefficient
    Q = np.zeros((Ng, Ng), dtype=np.float64)

    for i in range(Ng):
        for j in range(Ng):
            s = 0.0

            for k in range(Ng):
                if p_x[i] > 1e-12 and p_y[k] > 1e-12:
                    s += p[i, k] * p[j, k] / (p_x[i] * p_y[k])

            Q[i, j] = s

    eigvals = np.linalg.eigvals(Q)
    eigvals = np.real(eigvals)
    eigvals = np.sort(eigvals)[::-1]

    if len(eigvals) >= 2:
        h14 = np.sqrt(max(0.0, eigvals[1]))
    else:
        h14 = 0.0

    features = np.array([
        h1, h2, h3, h4, h5, h6, h7,
        h8, h9, h10, h11, h12, h13, h14
    ], dtype=np.float64)

    return features


# ------------------------------------------------------------
# 4.4. Tính GLCM và đặc trưng Haralick cho 4 hướng
# ------------------------------------------------------------

d = 1
angles = [0, 45, 90, 135]

feature_names = [
    "h1  - Angular Second Moment",
    "h2  - Contrast",
    "h3  - Correlation",
    "h4  - Variance",
    "h5  - Inverse Difference Moment",
    "h6  - Sum Average",
    "h7  - Sum Variance",
    "h8  - Sum Entropy",
    "h9  - Entropy",
    "h10 - Difference Variance",
    "h11 - Difference Entropy",
    "h12 - IMC1",
    "h13 - IMC2",
    "h14 - Max Correlation Coefficient"
]

haralick_4dirs = []

for angle in angles:
    print("\n" + "=" * 80)
    print(f"GLCM với d = {d}, theta = {angle}°")
    print("=" * 80)

    P = compute_glcm_count_matrix(
        I_level_idx=I_q_idx,
        d=d,
        theta_deg=angle,
        num_levels=5,
        symmetric=False
    )

    # Công thức/thuật toán minh họa:
    # - Chuẩn hóa ma trận đếm P(i,j) thành ma trận xác suất p(i,j).
    p = P / np.sum(P)

    print("\nP(i,j) - ma trận GLCM đếm:")
    print(P.astype(np.int64))

    print("\np(i,j) - ma trận xác suất GLCM:")
    np.set_printoptions(precision=6, suppress=True)
    print(p)

    features = haralick_14_features_from_p(p)
    haralick_4dirs.append(features)

    print("\n14 đặc trưng Haralick:")
    for name, value in zip(feature_names, features):
        print(f"{name:40s}: {value:.10f}")


# ------------------------------------------------------------
# 4.5. Tạo vector Haralick 28 chiều
# ------------------------------------------------------------

haralick_4dirs = np.array(haralick_4dirs)

mu_features = np.mean(haralick_4dirs, axis=0)
sigma_features = np.std(haralick_4dirs, axis=0, ddof=0)

F_GLCM_28 = []

for k in range(14):
    F_GLCM_28.append(mu_features[k])
    F_GLCM_28.append(sigma_features[k])

F_GLCM_28 = np.array(F_GLCM_28, dtype=np.float64)


# ============================================================
# 5. HIỂN THỊ KẾT QUẢ
# ============================================================

print("\n" + "=" * 80)
print("Vector Haralick 28 chiều")
print("=" * 80)
print(F_GLCM_28)

print("\nTừng cặp (mu_k, sigma_k):")
for k in range(14):
    print(f"h{k+1:02d}: mu = {mu_features[k]:.10f}, sigma = {sigma_features[k]:.10f}")


# ============================================================
# 6. KIỂM TRA KẾT QUẢ
# ============================================================

print("\n" + "=" * 80)
print("KIỂM TRA KẾT QUẢ")
print("=" * 80)

print("Số hướng đã tính:", haralick_4dirs.shape[0])
print("Số đặc trưng Haralick mỗi hướng:", haralick_4dirs.shape[1])
print("Kích thước vector Haralick cuối cùng:", F_GLCM_28.shape)

assert haralick_4dirs.shape == (4, 14)
assert F_GLCM_28.shape == (28,)
assert np.all(np.isfinite(F_GLCM_28))

print("Kiểm tra hoàn tất: vector Haralick 28 chiều hợp lệ.")


# ============================================================
# 7. GỢI Ý THỬ NGHIỆM CHO NGƯỜI HỌC
# ============================================================

# Có thể thay đổi d = 1 thành d = 2 hoặc d = 3 để quan sát ảnh hưởng của khoảng cách lân cận.
# Có thể đổi symmetric=False thành symmetric=True trong hàm compute_glcm_count_matrix.
# Có thể thay đổi số mức lượng tử hóa bằng cách sửa gray_levels và num_levels tương ứng.

Kích thước ảnh: (57, 59)
Các mức xám sau lượng tử hóa: [128 192 255]

GLCM với d = 1, theta = 0°

P(i,j) - ma trận GLCM đếm:
[[   0    0    0    0    0]
 [   0    0    0    0    0]
 [   0    0   36   96    0]
 [   0    0   97 3075    1]
 [   0    0    0    1    0]]

p(i,j) - ma trận xác suất GLCM:
[[0.       0.       0.       0.       0.      ]
 [0.       0.       0.       0.       0.      ]
 [0.       0.       0.010889 0.029038 0.      ]
 [0.       0.       0.029341 0.930127 0.000302]
 [0.       0.       0.       0.000302 0.      ]]

14 đặc trưng Haralick:
h1  - Angular Second Moment             : 0.8669591558
h2  - Contrast                          : 0.0589836661
h3  - Correlation                       : 0.2398835415
h4  - Variance                          : 0.0386597504
h5  - Inverse Difference Moment         : 0.9705081670
h6  - Sum Average                       : 5.9204476709
h7  - Sum Variance                      : 0.0962122618
h8  - Sum Entropy                       : 0.2869176